# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

In [12]:
def constant(n,m,c):
    return [[c for _ in range(m)] for _ in range(n)]
def zeroes(n,m):
    zeroes = [[0 for _ in range(m)] for _ in range(n)]
    ones = [[1 for _ in range(m)] for _ in range(n)]
    return zeroes, ones
def eye(n):
    identity = []
    for i in range(n):
        row = [0] * n
        row[i] = 1
        identity.append(row)
    return identity

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [24]:
A = matrix([[2, 7], [1, 5]])
B = matrix([[1, 2], [4, 5]])
C = matrix([[9, 10], [11, 12]])
I = matrix([[1, 0], [0, 1]])

# (AB)C = A(BC)
AB_C = (A * B) * C
A_BC = A * (B * C)
print("(AB)C == A(BC):", AB_C == A_BC)

# A(B+C) = AB+AC
A_BplusC = A * (B + C)
AB_AC = (A * B) + (A * C)
print("A(B+C) == AB+AC:", A_BplusC == AB_AC)

# AB != BA
AB = A * B
BA = B * A
print("AB == BA:", AB == BA)  # should print False

# AI = A
AI = A * I
print("AI == A:", AI == A)


(AB)C == A(BC): True
A(B+C) == AB+AC: True
AB == BA: False
AI == A: True


In [22]:
class matrix (object):
    def __init__(self, *args):
        if len(args) == 2 and isinstance(args[0], int) and isinstance(args[1], int):
            if args[0] < 1 or args[1] < 1:
                raise ValueError("matrix dimensions must be positive integers")
            self.n = args[0]
            self.m = args[1]
            self.data = [[0 for _ in range(self.m)] for _ in range(self.n)]
        
        elif len(args) == 1 and isinstance(args[0], list):
            data = args[0]
            if len(data) == 0 or not all(isinstance(row, list) for row in data):
                raise ValueError("matrix data must be a non-empty list of lists")
            row_len = len(data[0])
            if row_len == 0 or not all(len(row) == row_len for row in data):
                raise ValueError("all rows in the matrix must have the same number of columns")
            self.n = len(data)
            self.m = row_len
            self.data = [row[:] for row in data]

        else:
            raise ValueError("matrix can only be intialized with n,m or a list of lists")
        
    def __getitem__(self, key, row_start=None, row_end=None, col_start=None, col_end=None):
        if isinstance(key, int):
            return self.data[key]
        elif isinstance(key, tuple) and len(key) == 2:
            i, j = key
            return self.data[i][j]
        elif row_start is not None and row_end is not None and col_start is not None and col_end is not None:
            return self.block(row_start, row_end, col_start, col_end)
        raise TypeError("invalid key type for matrix indexing")
    
    def __setitem__(self, key, value):
        if isinstance(key, int):
            self.data[key] = value
        elif isinstance(key, tuple) and len(key) == 2:
            i, j = key
            self.data[i][j] = value
        else:
            raise TypeError("invalid key type for matrix indexing")
            
    def assign(self, other):
        if not isinstance(other, matrix):
            raise ValueError("can only assign from another matrix")
        if self.n != other.n or self.m != other.m:
            raise ValueError("matrix dimensions must match for assignment")
        for i in range(self.n):
            for j in range(self.m):
                self.data[i][j] = other.data[i][j]

    def shape(self):
        return (self.n, self.m)
    
    def transpose(self):
        transposed_data = [[self.data[i][j] for i in range(self.n)] for j in range(self.m)]
        return matrix(transposed_data)
    
    def row_return(self, index):
        if index < 0 or index >= self.n:
            raise IndexError("row index out of range")
        return self.data[index]
    
    def column_return(self, index):
        if index < 0 or index >= self.m:
            raise IndexError("column index out of range")
        return [self.data[i][index] for i in range(self.n)]
    
    def to_list(self):
        return [[0 for _ in range(self.m)] for _ in range(self.n)]
        
    def block(self, row_start, row_end, col_start, col_end):
        if row_start < 0 or row_end > self.n or col_start < 0 or col_end > self.m:
            raise IndexError("block indices out of range")
        block_data = [self.data[i][col_start:col_end] for i in range(row_start, row_end)]
        return matrix(block_data)
    
    def __add__(self, other):
        if not isinstance(other, matrix):
            raise ValueError("can only add another matrix")
        if self.n != other.n or self.m != other.m:
            raise ValueError("matrix dimensions must match for addition")
        added_data = [[self.data[i][j] + other.data[i][j] for j in range(self.m)] for i in range(self.n)]
        return matrix(added_data)
    
    def __sub__(self, other):
        if not isinstance(other, matrix):
            raise ValueError("can only add another matrix")
        if self.n != other.n or self.m != other.m:
            raise ValueError("matrix dimensions must match for addition")
        subtracted_data = [[self.data[i][j] - other.data[i][j] for j in range(self.m)] for i in range(self.n)]
        return matrix(subtracted_data)
    
    def __mul__(self, other):
        if isinstance(other, int) or isinstance(other, float):
            scalar_data = [[self.data[i][j] * scalar for j in range(self.m)] for i in range(self.n)]
            return matrix(scalar_data)
        elif isinstance(other, matrix):
            if self.m != other.n:
                raise ValueError("number of columns of the first matrix must equal number of rows of the second matrix for multiplication")
            multiplied_data = [[sum(self.data[i][k] * other.data[k][j] for k in range(self.m)) for j in range(other.m)] for i in range(self.n)]
            return matrix(multiplied_data)
        
    def __rmul__(self, other): 
        return self.__mul__(other)
    
    def linear_multiply(self, other):
        if not isinstance(other, matrix):
            raise ValueError("can only multiply by another matrix")
        if self.m != other.m or self.n != other.n:
            raise ValueError("matrix dimensions must match for linear multiplication")
        linear_multiplied_data = [[self.data[i][j] * other.data[i][j] for j in range(self.m)] for i in range(self.n)]
        return matrix(linear_multiplied_data)
    
    def __eq__(self, other):
        if not isinstance(other, matrix):
            raise ValueError("can only compare with another matrix")
        return self.n == other.n and self.m == other.m and all(self.data[i][j] == other.data[i][j] for i in range(self.n) for j in range(self.m))
    
